# Kaggriculture: от top replay до Kaggle submission

Эта тетрадка выполняет весь рабочий процесс: подключает Drive, проверяет Kaggle, скачивает replay топ-игроков, строит Worker Dataset, обучает модель, проверяет готовый архив и отправляет его в соревнование.

Перед запуском добавьте токен в **Colab → Secrets** под именем `KAGGLE_API_TOKEN` и разрешите этой тетрадке доступ к нему. Нажмите **Runtime → Run all**. Запросы разрешений появятся в самом начале. Тяжёлые этапы сохраняются на Google Drive и будут использованы повторно после перезапуска Colab.

In [ ]:
# Единственное место, где нужно менять параметры запуска.
TOP_PLAYERS = 10
REPLAYS_PER_PLAYER = 50
EPOCHS = 3
BATCH_SIZE = 1024
BEST_SUBMISSION_ONLY = True
WINNER_ONLY = True
SUBMIT_TO_KAGGLE = True
FORCE_SUBMIT = False
SUBMISSION_MESSAGE = "Teacher worker behavior cloning v1"

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/Kaggriculture")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Google Drive подключён: {DRIVE_ROOT}")

In [ ]:
# Получаем свежий код, ставим CLI и проверяем авторизацию до долгой работы.
import os
import subprocess
import sys
from google.colab import userdata

REPOSITORY = "https://github.com/GrigoriiIurev/Kaggriculture.git"
PROJECT = Path("/content/Kaggriculture")
if (PROJECT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
os.chdir(PROJECT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "kaggle", "kaggle-environments"],
    check=True,
)
try:
    kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    kaggle_token = None
if kaggle_token:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token

auth_check = subprocess.run(
    ["kaggle", "competitions", "list", "-s", "kaggriculture"],
    text=True, capture_output=True,
)
if auth_check.returncode != 0:
    raise RuntimeError(
        "Kaggle не авторизован. Добавьте KAGGLE_API_TOKEN в Colab Secrets.\n"
        + auth_check.stderr
    )
entered = subprocess.run(
    ["kaggle", "competitions", "list", "--group", "entered"],
    text=True, capture_output=True, check=True,
)
if "kaggriculture" not in entered.stdout.lower():
    raise RuntimeError(
        "Аккаунт не присоединился к Kaggriculture. Откройте страницу соревнования "
        "на Kaggle и нажмите Join Competition."
    )
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Kaggle авторизован, правила приняты, код: {commit}")

## Полный автоматический запуск

Эта ячейка может работать несколько часов. Логи появляются на каждом этапе. При повторном запуске готовые replay, датасет, модель и уже отправленный идентичный архив используются повторно.

In [ ]:
command = [
    sys.executable, "-u", "run_colab_pipeline.py",
    "--top-players", str(TOP_PLAYERS),
    "--replays-per-player", str(REPLAYS_PER_PLAYER),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--drive-root", str(DRIVE_ROOT),
    "--message", SUBMISSION_MESSAGE,
]
if not BEST_SUBMISSION_ONLY:
    command.append("--all-submissions")
if not WINNER_ONLY:
    command.append("--both-players")
if SUBMIT_TO_KAGGLE:
    command.append("--submit")
if FORCE_SUBMIT:
    command.append("--force-submit")

print("Запускаю полный pipeline...", flush=True)
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
# Показываем итоговые файлы и текущее состояние отправки.
dataset_id = (
    f"top{TOP_PLAYERS}_replays{REPLAYS_PER_PLAYER}_"
    f"best{int(BEST_SUBMISSION_ONLY)}_winner{int(WINNER_ONLY)}"
)
run_id = f"{dataset_id}_epochs{EPOCHS}_batch{BATCH_SIZE}"
RESULTS = DRIVE_ROOT / "results" / run_id
print(f"\nГотовые файлы: {RESULTS}")
for path in sorted(RESULTS.iterdir()):
    print(f"- {path.name}: {path.stat().st_size / 1024 / 1024:.2f} MB")

if SUBMIT_TO_KAGGLE:
    subprocess.run(
        ["kaggle", "competitions", "submissions", "kaggriculture"],
        check=True,
    )